# Recorded crime (Y1): the SPS extract, June–August 2026

Read from `Data/sps_crime_2026-06-01_2026-08-31/02_cleaned/`, the output of
`Scripts/crime_data/clean_sps_crime.py`; nothing is recomputed or written.

1. What the extract holds
2. The study categories, city-wide and inside the CBD
3. By month
4. Patterns flagged in the cleaned file

Until SPS's terms of use are recorded, only aggregated counts leave the team (`Docs/data_ethics.md`); this
notebook shows counts only.

In [ ]:
# Shared folders, project modules and plot settings
from _setup import *
# Show plots inside the notebook
%matplotlib inline
# Reading the cleaning script's JSON summary
import json

## Step 1: What the extract holds

In [ ]:
# The cleaning script's own summary of the extract
summary = json.loads((SPS / "02_cleaned" / "cleaning_report.json").read_text())
# Records, the time span, and how many fall in each decision group
print(f"{summary['raw_rows']:,} offence records, reported {summary['reported_at_min']} to {summary['reported_at_max']}")
print("by decision:", summary["rows_by_decision"])
print("fields the extract does not have:", summary["fields_absent_from_extract"])

## Step 2: The study categories

Five categories are included and assault is conditional (outdoor assaults can't be separated: the extract has no premise type). Offence records count every offence; incidents merge records with the same time and place.

In [ ]:
# One row per study category: records and incidents, city-wide and inside the CBD
categories = pd.read_csv(SPS / "02_cleaned" / "category_counts.csv")
categories[["study_category", "category_decision", "crime_group", "offence_rows", "incidents", "offence_rows_cbd"]]

In [ ]:
# Two panels: city-wide and inside the CBD (very different scales)
fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
left.barh(categories["study_category"], categories["offence_rows"], color="#2a78d6")
right.barh(categories["study_category"], categories["offence_rows_cbd"], color="#eb6834")
# Titles and labels for both panels
left.set_title("City-wide", loc="left")
right.set_title("Inside the CBD polygon", loc="left")
left.set_xlabel("offence records")
right.set_xlabel("offence records")
left.invert_yaxis()
plt.tight_layout()
plt.show()

## Step 3: By month

In [ ]:
# The cleaned rows: one per offence record
rows = pd.read_csv(SPS / "02_cleaned" / "sps_offences_2026-06-01_2026-08-31_clean.csv")
# Only the study categories (the included ones, and assault)
study = rows[rows["category_decision"].isin(["included", "conditional"])]
# Records per category and month, with totals
by_month = study.pivot_table(index="study_category", columns="month", values="record_id", aggfunc="count",
                             fill_value=0, margins=True, margins_name="total")
by_month

## Step 4: Patterns flagged in the cleaned file

These rows are flagged, not removed. The fixed-address midnight records are all "Other Criminal Code" at three addresses, so they stay out of any "all crime" count for the CBD.

In [ ]:
# How many rows carry each flag
flags = pd.Series({"fixed-address midnight records": int(rows["fixed_address_midnight"].sum()),
                   "records repeating another record exactly": int((rows["identical_rank"] > 1).sum()),
                   "records with no neighbourhood": int(rows["neighbourhood_missing"].sum()),
                   "records inside the CBD polygon": int(rows["in_cbd"].sum())}, name="records")
flags.to_frame()

## Takeaway

- Recorded crime is shaped by reporting and enforcement: describe it as recorded crime.
- Locations are hundred-blocks or intersections, so crime is counted per street segment, not per sample
  point (`Docs/crime_data.md`).